# M1 Lab — Orientation & Intro to Data Science

**Dataset:** `sales_monthly.csv` &nbsp;|&nbsp; **Anchors:** McKinney Ch1, Géron Ch1

No `[AI-OFF]` cells in M1. Use this lab to set up your environment and practice the disclosure workflow.

## L1.4 — Reproducibility setup `[Expert]`

In [ ]:
import numpy as np, random, pandas as pd
np.random.seed(42)
random.seed(42)

# Notebook header — copy to every future notebook
# Author: [your name]
# Dataset: sales_monthly.csv
# Environment: ds-gemma-course

## L1.3 — Load and inspect `[Expert]`

In [ ]:
df = pd.read_csv('sales_monthly.csv')
print(df.shape)
df.head()

## L1.3 — Pipeline questions (write in markdown)

Answer in the markdown cell below:
1. What question does this dataset help us answer?
2. What cleaning steps might be needed (guess before looking)?
3. What would a useful insight look like?

*(your answers here)*

## L1.5 — Correlation vs causation discussion `[Expert]`

Pick any two columns that look related. In a markdown cell, propose:
- The correlation you would expect
- A plausible **third** variable that could be the actual cause

*(your hypothesis here)*

---
## Submission checklist
- [ ] All cells run from a fresh kernel
- [ ] `AI_USE.md` initialised (even if empty for M1)
- [ ] Reproducibility block present in cell 1

## L1.7 — Local AI Workspace Setup


### Step 1: Ollama health check

Run this cell first. It verifies that Ollama is running. If you get a connection error, make sure Docker Compose is running (or Ollama is started on your machine).


In [ ]:
import requests

def check_ollama():
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=5)
        models = r.json().get("models", [])
        if models:
            print("✅ Ollama is running!")
            print("Available models:")
            for m in models:
                print(f"  - {m['name']}")
        else:
            print("⚠️  Ollama is running but no models are pulled yet.")
            print("   Run: ollama pull gemma4n")
    except requests.exceptions.ConnectionError:
        print("❌ Cannot connect to Ollama at localhost:11434")
        print("   Make sure Ollama is running (docker compose up or ollama serve)")

check_ollama()


### Step 2: First chat completion (non-streaming)

This sends a prompt to Gemma 4n and waits for the full response. Simpler to debug than streaming.


In [ ]:
import requests, json

def ask_gemma(prompt: str, model: str = "gemma4n") -> str:
    """Send a prompt to the local Gemma model. Returns the response text."""
    r = requests.post(
        "http://localhost:11434/api/chat",
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
            "stream": False
        },
        timeout=120
    )
    r.raise_for_status()
    return r.json()["message"]["content"]

# Your first local AI call!
response = ask_gemma("Explain the data-to-insight pipeline in 3 sentences for a beginner.")
print(response)


### Step 3: Streaming output (token by token)

Streaming shows the response being generated in real time — this is how the Gemma chat in the sidebar works.


In [ ]:
def ask_gemma_streaming(prompt: str, model: str = "gemma4n") -> None:
    """Send a prompt and print each token as it arrives."""
    r = requests.post(
        "http://localhost:11434/api/chat",
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
            "stream": True
        },
        stream=True,
        timeout=120
    )
    r.raise_for_status()
    for line in r.iter_lines():
        if line:
            chunk = json.loads(line)
            if not chunk.get("done", False):
                print(chunk["message"]["content"], end="", flush=True)
    print()  # newline at end

ask_gemma_streaming("What makes a data science question a good one? Answer in 2-3 sentences.")


### Step 4: Log your first AI interaction

Open (or create) `AI_USE.md` in your project root and add an entry using the disclosure template from M1 prompts.md.
